# Deploy Canary
Despliega la versión challenger en el endpoint con `traffic_percent`% del tráfico (el resto al champion). Idempotente: reescribe el config del endpoint con el split indicado y asigna el alias `challenger`. Serverless + scale-to-zero.

In [0]:
%pip install -q databricks-sdk mlflow

In [0]:
import time, mlflow
from mlflow.tracking import MlflowClient
from databricks.sdk import WorkspaceClient
from databricks.sdk.service import serving
mlflow.set_registry_uri("databricks-uc")

MODEL_NAME = dbutils.widgets.get("model_name")
MODEL_VERSION = dbutils.widgets.get("model_version")
ENDPOINT_NAME = dbutils.widgets.get("endpoint_name")
TRAFFIC_PERCENT = int(dbutils.widgets.get("traffic_percent"))

client = MlflowClient()
w = WorkspaceClient()

# alias challenger -> esta version (idempotente)
client.set_registered_model_alias(MODEL_NAME, "challenger", MODEL_VERSION)

# version del champion actual (si existe)
champion_version = None
try:
    champion_version = client.get_model_version_by_alias(MODEL_NAME, "champion").version
except Exception:
    champion_version = None

def se(name, version):
    return serving.ServedEntityInput(name=name, entity_name=MODEL_NAME,
                                     entity_version=str(version),
                                     workload_size="Small",
                                     scale_to_zero_enabled=True)

served = [se("challenger", MODEL_VERSION)]
routes = [serving.Route(served_model_name="challenger", traffic_percentage=str(TRAFFIC_PERCENT))]
if champion_version is not None:
    served.append(se("champion", champion_version))
    routes.append(serving.Route(served_model_name="champion", traffic_percentage=str(100 - TRAFFIC_PERCENT)))

w.serving_endpoints.update_config(name=ENDPOINT_NAME, served_entities=served,
                                  traffic_config=serving.TrafficConfig(routes=routes))

# esperar a READY (scale-to-zero: puede tardar en arrancar; no bloquea indefinidamente)
for _ in range(60):
    st = w.serving_endpoints.get(ENDPOINT_NAME)
    if getattr(st.state, "ready", None) == "READY":
        break
    time.sleep(10)

rest = 100 - TRAFFIC_PERCENT if champion_version else 100
print(f"Endpoint {ENDPOINT_NAME} actualizado: challenger={TRAFFIC_PERCENT}% champion={rest}% (champion_version={champion_version})")